# Media Framing: Tagesschau Inspection First

This notebook is intentionally compact.

It does four things:
- loads the older GPT-coded results if that CSV is available
- reproduces the old article-sampling logic for 50 Tagesschau articles
- applies the codebook synchronously to those Tagesschau context rows using the same result-row structure as before
- prepares the full batch files only after the Tagesschau inspection is done

## 1. Setup

In [23]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR_CANDIDATES = [
    Path.cwd() / "2a_NER",
    Path.cwd(),
    Path.cwd().parent / "2a_NER",
    Path("/Users/MattisHaumann/Dev/Thesis/2a_NER"),
]
NOTEBOOK_DIR = next(
    (path for path in NOTEBOOK_DIR_CANDIDATES if (path / "media_framing_batch_utils.py").exists()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("Could not locate 2a_NER/media_framing_batch_utils.py from the current working directory.")

PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(NOTEBOOK_DIR))

from media_framing_batch_utils import (
    build_batch_requests_df,
    build_hit_input,
    compile_master_pattern,
    estimate_batch_cost,
    estimate_tokens_heuristic,
    extract_media_contexts,
    extract_output_text_from_responses_body,
    read_env_value,
    sort_by_source_order,
    validate_batch_requests,
    write_batch_jsonl,
    write_manifest_csv,
)

DATA_PATH = NOTEBOOK_DIR / "df_combined.csv"
PROMPT_PATH = NOTEBOOK_DIR / "framing_codebook_prompt.txt"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "batch_media_framing"
LEGACY_RESULTS_CANDIDATES = [
    NOTEBOOK_DIR / "outputs" / "media_framing_gpt5mini_entity_context_results_v2.csv",
    NOTEBOOK_DIR / "outputs" / "media_framing_gpt5mini_entity_context_results.csv",
]
TAGESSCHAU_SYNC_RESULTS_PATH = OUTPUT_DIR / "media_framing_tagesschau_50_sync_results.csv"
TAGESSCHAU_SYNC_ERRORS_PATH = OUTPUT_DIR / "media_framing_tagesschau_50_sync_errors.csv"
TAGESSCHAU_50_BATCH_JSONL_PATH = OUTPUT_DIR / "media_framing_tagesschau_50_batch.jsonl"
TAGESSCHAU_50_MANIFEST_PATH = OUTPUT_DIR / "media_framing_tagesschau_50_manifest.csv"
FULL_BATCH_JSONL_PATH = OUTPUT_DIR / "media_framing_full_batch.jsonl"
FULL_MANIFEST_PATH = OUTPUT_DIR / "media_framing_full_manifest.csv"

MODEL_NAME = "gpt-5-mini"
WINDOW = 1
RANDOM_STATE = 42
TAGESSCHAU_ARTICLE_SAMPLE_N = 50

FRAME_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "category": {
            "type": "string",
            "enum": [
                "POSITIONS-/PARTEILICHKEITS-BIAS",
                "VERZERRUNG/MANIPULATION",
                "DISINFORMATION/FALSCHDARSTELLUNG",
                "VERSAGEN/INKOMPETENZ",
                "NEUTRAL",
                "IRRELEVANT",
            ],
        },
        "evidence": {"type": "string"},
    },
    "required": ["category", "evidence"],
}

ANALYSIS_INSTRUCTIONS = (
    "Return valid JSON that matches the schema exactly. "
    "Do not add any keys beyond category and evidence."
)

for required_path in [DATA_PATH, PROMPT_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required file not found: {required_path}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LEGACY_RESULTS_PATH = next((path for path in LEGACY_RESULTS_CANDIDATES if path.exists()), None)

print(f"Data path: {DATA_PATH}")
print(f"Prompt path: {PROMPT_PATH}")
print(f"Legacy results path: {LEGACY_RESULTS_PATH}")

Data path: /Users/MattisHaumann/Dev/Thesis/2a_NER/df_combined.csv
Prompt path: /Users/MattisHaumann/Dev/Thesis/2a_NER/framing_codebook_prompt.txt
Legacy results path: None


## 2. Load Corpus and Optional Older Results

In [24]:
df = pd.read_csv(DATA_PATH)

if "row_id" not in df.columns:
    df = df.reset_index().rename(columns={"index": "row_id"})

required_columns = {"row_id", "source", "Title", "Text"}
missing_columns = sorted(required_columns.difference(df.columns))
if missing_columns:
    raise ValueError(f"df_combined.csv is missing required columns: {missing_columns}")

search_columns = [column for column in ["row_id", "source", "Date", "Title", "Text"] if column in df.columns]
search_df = df[search_columns].copy()
search_df["source"] = search_df["source"].fillna("").astype(str)

CODEBOOK_PROMPT = PROMPT_PATH.read_text(encoding="utf-8").strip()
if not CODEBOOK_PROMPT:
    raise ValueError(f"Prompt file is empty: {PROMPT_PATH}")

legacy_results_df = pd.DataFrame()
legacy_category_summary_df = pd.DataFrame()
if LEGACY_RESULTS_PATH is not None:
    legacy_results_df = pd.read_csv(LEGACY_RESULTS_PATH)
    if "source" in legacy_results_df.columns:
        legacy_results_df = legacy_results_df[legacy_results_df["source"] != "Tagesschau"].copy()
    if "category" in legacy_results_df.columns and "source" in legacy_results_df.columns:
        legacy_category_summary_df = (
            legacy_results_df.groupby(["source", "category"], as_index=False)
            .agg(coded_contexts=("row_id", "size"))
        )
        legacy_category_summary_df["share_pct"] = (
            legacy_category_summary_df.groupby("source")["coded_contexts"]
            .transform(lambda values: (values / values.sum() * 100).round(2))
        )
        legacy_category_summary_df = legacy_category_summary_df.pipe(sort_by_source_order)

print(f"Articles loaded: {len(search_df):,}")
print(f"Unique sources: {search_df['source'].nunique():,}")
print(f"Legacy coded rows available: {len(legacy_results_df):,}")
if not legacy_category_summary_df.empty:
    display(legacy_category_summary_df)

Articles loaded: 20,440
Unique sources: 7
Legacy coded rows available: 0


## 3. Extract Contexts and Build the 50-Article Tagesschau Sample

This keeps the same logic as before:
- sample hit-positive articles first
- expand them to retained context windows
- classify one merged context window at a time

The direct Tagesschau boilerplate filter is already applied at extraction time.

In [25]:
MASTER_PATTERN = compile_master_pattern()

extraction = extract_media_contexts(search_df, pattern=MASTER_PATTERN, window=WINDOW)
media_context_df = extraction["media_context_df"]
media_article_df = extraction["media_article_df"]
kept_hits_df = extraction["kept_hits_df"]
excluded_hits_df = extraction["excluded_hits_df"]

tagesschau_article_pool_df = media_article_df.loc[media_article_df["source"] == "Tagesschau"].copy()
if tagesschau_article_pool_df.empty:
    raise ValueError("No Tagesschau articles remain after extraction.")

tagesschau_50_articles_df = (
    tagesschau_article_pool_df.sample(
        n=min(TAGESSCHAU_ARTICLE_SAMPLE_N, len(tagesschau_article_pool_df)),
        random_state=RANDOM_STATE,
    )
    .sort_values("row_id")
    .reset_index(drop=True)
)

tagesschau_row_ids = set(tagesschau_50_articles_df["row_id"].tolist())
tagesschau_50_context_df = media_context_df.loc[media_context_df["row_id"].isin(tagesschau_row_ids)].copy()

tagesschau_50_hits_df = tagesschau_50_context_df.copy()
tagesschau_50_hits_df["hit_id"] = tagesschau_50_hits_df.apply(
    lambda row: f"{row['row_id']}|{row['context_idx']}|{row['context_window']}",
    axis=1,
).map(lambda value: __import__('hashlib').md5(value.encode('utf-8')).hexdigest()[:16])

tagesschau_50_hits_df = tagesschau_50_hits_df[
    [column for column in [
        "hit_id",
        "row_id",
        "source",
        "Title",
        "hit_text",
        "context_idx",
        "count_hits",
        "count_unique_entities",
        "context_window",
    ] if column in tagesschau_50_hits_df.columns]
].copy()

tagesschau_50_hit_summary_df = (
    kept_hits_df.loc[kept_hits_df["row_id"].isin(tagesschau_row_ids)]
    .groupby("normalized_hit", as_index=False)
    .agg(matches=("row_id", "size"), unique_articles=("row_id", "nunique"))
    .sort_values(["matches", "unique_articles", "normalized_hit"], ascending=[False, False, True])
)

tagesschau_50_excluded_summary_df = (
    excluded_hits_df.loc[excluded_hits_df["row_id"].isin(tagesschau_row_ids)]
    .groupby(["normalized_hit", "exclusion_reason"], as_index=False)
    .agg(excluded_matches=("row_id", "size"), unique_articles=("row_id", "nunique"))
    .sort_values(["excluded_matches", "unique_articles", "normalized_hit"], ascending=[False, False, True])
)

residual_broadcaster_terms = ["MDR", "SWR", "NDR Info", "NDR", "WDR", "BR", "rbb"]
tagesschau_residual_term_preview_df = tagesschau_50_context_df.loc[
    tagesschau_50_context_df["hit_text"].fillna("").apply(
        lambda text: any(term in text.split(" | ") for term in residual_broadcaster_terms)
    ),
    [column for column in ["row_id", "Title", "hit_text", "context_window"] if column in tagesschau_50_context_df.columns],
].head(12)

print(f"Tagesschau sampled articles: {len(tagesschau_50_articles_df):,}")
print(f"Tagesschau sampled context rows for GPT: {len(tagesschau_50_hits_df):,}")
display(tagesschau_50_hit_summary_df.head(15))
display(tagesschau_50_excluded_summary_df)
display(tagesschau_residual_term_preview_df)

Tagesschau sampled articles: 50
Tagesschau sampled context rows for GPT: 67


,normalized_hit,matches,unique_articles
1,Deutschlandfunk,18,18
21,dpa,8,7
4,MDR,8,3
12,SWR,7,4
6,NDR Info,6,6
5,NDR,4,4
0,BR,4,2
3,Handelsblatt,3,1
10,Reuters,2,2
18,WDR,2,2


,normalized_hit,exclusion_reason,excluded_matches,unique_articles
0,ARD,tagesschau_self_reference,29,21
2,Tagesschau,tagesschau_self_reference,7,6
1,Das Erste,tagesschau_self_reference,7,5
3,Tagesschau24,tagesschau_self_reference,5,5


,row_id,Title,hit_text,context_window
7660,14224,"""Eine verfassungswidrige Invasion""",NDR Info,"Und dann ergänzt er noch: ""Wir müssen unsere S..."
8406,15644,Ein paar Tage Hochsommer in Deutschland,SWR,Besonders den Menschen in Baden-Württemberg st...
8407,15644,Ein paar Tage Hochsommer in Deutschland,NDR,In der Nacht zu Donnerstag kann es allerdings ...
8408,15644,Ein paar Tage Hochsommer in Deutschland,SWR,Laut DWD werden am Donnerstag in Teilen Nordba...
8714,15954,Eiertanz um Fleischfakten,NDR,Konsumverhalten und Klima Warum hat Agrarminis...
9030,16242,"Gutachten sieht ""mangelhaftes Sicherheitskonzept""",MDR,Die Amokfahrt auf dem Magdeburger Weihnachtsma...
9031,16242,"Gutachten sieht ""mangelhaftes Sicherheitskonzept""",MDR,Unter den üblichen Standards Das Sicherheitsko...
9032,16242,"Gutachten sieht ""mangelhaftes Sicherheitskonzept""",MDR,Komme ein solches nicht-zertifiziertes System ...
9033,16242,"Gutachten sieht ""mangelhaftes Sicherheitskonzept""",MDR,Ebenso ist der Abschlussbericht der Sonderkomm...
9034,16242,"Gutachten sieht ""mangelhaftes Sicherheitskonzept""",MDR,Der Zwischenbericht werde zunächst in der Verw...


## 4. Apply GPT to the Tagesschau Sample Using the Old Result-Row Shape

This writes the same kind of result rows the older synchronous notebook wrote:
- `hit_id`
- `row_id`
- `source`
- `Title`
- `hit_text`
- `context_idx`
- `count_hits`
- `count_unique_entities`
- `context_window`
- `model`
- `response_id`
- `category`
- `evidence`
- `raw_response_json`

That makes the Tagesschau inspection directly comparable in structure to the earlier non-Tagesschau run.

In [26]:
preview_input = build_hit_input(CODEBOOK_PROMPT, tagesschau_50_hits_df.iloc[0].to_dict())
print(f"Rows ready for synchronous GPT inspection: {len(tagesschau_50_hits_df):,}")
print(f"Results file: {TAGESSCHAU_SYNC_RESULTS_PATH}")
print(f"Errors file: {TAGESSCHAU_SYNC_ERRORS_PATH}")
print(preview_input[:900])

Rows ready for synchronous GPT inspection: 67
Results file: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/media_framing_tagesschau_50_sync_results.csv
Errors file: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/media_framing_tagesschau_50_sync_errors.csv
Sie sind Experte darin, zu erkennen, wie Medienunternehmen in deutschsprachigen Nachrichtenartikeln dargestellt werden. Analysieren Sie die folgende Erwähnungen der Medienakteure im Kontext und ordnen Sie die Art des Vorwurfs der Voreingenommenheit (falls vorhanden), der gegen die Medienakteure erhoben wird, anhand der untenstehenden Taxonomie ein.

Taxonomie:

POSITIONS-/PARTEILICHKEITS-BIAS
Der Medienakteur wird beschuldigt, bestimmten Interessen zu dienen oder politische Loyalitäten zu haben.
Umfasst: Dienst an politischen Parteien/Politikern/Regierungen; Förderung bestimmter Ideologien (links/rechts/globalistisch/Establishment); systematische Bevorzugung bestimmter sozialer Gruppen; Kontroll

In [27]:
import requests
from pandas.errors import EmptyDataError

RUN_TAGESSCHAU_SYNC = False
SAVE_EVERY = 10

api_key, api_key_source = read_env_value("OPENAI_API_KEY", project_root=PROJECT_ROOT)
if not api_key:
    raise RuntimeError("OPENAI_API_KEY not found. Add it to .env or export it in your shell.")

if TAGESSCHAU_SYNC_RESULTS_PATH.exists() and TAGESSCHAU_SYNC_RESULTS_PATH.stat().st_size > 0:
    try:
        existing_results_df = pd.read_csv(TAGESSCHAU_SYNC_RESULTS_PATH)
    except EmptyDataError:
        existing_results_df = pd.DataFrame()
else:
    existing_results_df = pd.DataFrame()

processed_hit_ids = set()
if not existing_results_df.empty and "hit_id" in existing_results_df.columns:
    processed_hit_ids.update(existing_results_df["hit_id"].astype(str))

run_hits_df = tagesschau_50_hits_df[
    ~tagesschau_50_hits_df["hit_id"].astype(str).isin(processed_hit_ids)
].copy()

print(f"API key source: {api_key_source}")
print(f"Already processed rows: {len(processed_hit_ids):,}")
print(f"Rows remaining: {len(run_hits_df):,}")

if RUN_TAGESSCHAU_SYNC and not run_hits_df.empty:
    session = requests.Session()
    session.headers.update(
        {
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        }
    )

    results = existing_results_df.to_dict("records") if not existing_results_df.empty else []
    errors = []

    def save_progress(results_records, error_records):
        pd.DataFrame(results_records).to_csv(TAGESSCHAU_SYNC_RESULTS_PATH, index=False, encoding="utf-8")
        pd.DataFrame(error_records).to_csv(TAGESSCHAU_SYNC_ERRORS_PATH, index=False, encoding="utf-8")

    for idx, row in enumerate(run_hits_df.to_dict("records"), start=1):
        payload = {
            "model": MODEL_NAME,
            "instructions": ANALYSIS_INSTRUCTIONS,
            "input": build_hit_input(CODEBOOK_PROMPT, row),
            "text": {
                "format": {
                    "type": "json_schema",
                    "name": "media_bias_frame_v2",
                    "strict": True,
                    "schema": FRAME_SCHEMA,
                }
            },
        }

        try:
            response = session.post("https://api.openai.com/v1/responses", json=payload, timeout=180)
            response.raise_for_status()
            response_json = response.json()
            parsed_output = json.loads(extract_output_text_from_responses_body(response_json))

            results.append(
                {
                    "hit_id": row["hit_id"],
                    "row_id": row["row_id"],
                    "source": row["source"],
                    "Title": row["Title"],
                    "hit_text": row["hit_text"],
                    "context_idx": row.get("context_idx", ""),
                    "count_hits": row.get("count_hits", ""),
                    "count_unique_entities": row.get("count_unique_entities", ""),
                    "context_window": row["context_window"],
                    "model": MODEL_NAME,
                    "response_id": response_json.get("id", ""),
                    "category": parsed_output["category"],
                    "evidence": parsed_output["evidence"],
                    "raw_response_json": json.dumps(response_json, ensure_ascii=False),
                }
            )
        except requests.HTTPError as exc:
            try:
                error_body = exc.response.json()
            except ValueError:
                error_body = {"message": exc.response.text[:2000] if exc.response is not None else str(exc)}

            errors.append(
                {
                    "hit_id": row["hit_id"],
                    "row_id": row["row_id"],
                    "source": row["source"],
                    "Title": row["Title"],
                    "hit_text": row["hit_text"],
                    "context_window": row["context_window"],
                    "status_code": exc.response.status_code if exc.response is not None else None,
                    "error": json.dumps(error_body, ensure_ascii=False),
                }
            )
        except (requests.RequestException, json.JSONDecodeError, KeyError, ValueError) as exc:
            errors.append(
                {
                    "hit_id": row["hit_id"],
                    "row_id": row["row_id"],
                    "source": row["source"],
                    "Title": row["Title"],
                    "hit_text": row["hit_text"],
                    "context_window": row["context_window"],
                    "status_code": None,
                    "error": str(exc),
                }
            )

        if idx % SAVE_EVERY == 0 or idx == len(run_hits_df):
            save_progress(results, errors)
            print(f"Processed {idx}/{len(run_hits_df)} rows")
else:
    print("Synchronous GPT run disabled. Set RUN_TAGESSCHAU_SYNC = True only when you want to execute the Tagesschau inspection.")

API key source: /Users/MattisHaumann/Dev/Thesis/.env
Already processed rows: 67
Rows remaining: 0
Synchronous GPT run disabled. Set RUN_TAGESSCHAU_SYNC = True only when you want to execute the Tagesschau inspection.


## 5. Inspect the Tagesschau GPT Results

In [28]:
if TAGESSCHAU_SYNC_RESULTS_PATH.exists() and TAGESSCHAU_SYNC_RESULTS_PATH.stat().st_size > 0:
    tagesschau_results_df = pd.read_csv(TAGESSCHAU_SYNC_RESULTS_PATH)
    if "evidence" in tagesschau_results_df.columns:
        tagesschau_results_df["evidence"] = tagesschau_results_df["evidence"].fillna("")

    tagesschau_label_summary_df = (
        tagesschau_results_df.groupby("category", as_index=False)
        .agg(coded_contexts=("row_id", "size"), unique_articles=("row_id", "nunique"))
        .sort_values(["coded_contexts", "unique_articles", "category"], ascending=[False, False, True])
    )
    tagesschau_label_summary_df["share_pct"] = (
        tagesschau_label_summary_df["coded_contexts"] / tagesschau_label_summary_df["coded_contexts"].sum() * 100
    ).round(2)

    inspection_cols = [
        column for column in [
            "row_id",
            "source",
            "Title",
            "hit_text",
            "category",
            "evidence",
            "context_window",
        ] if column in tagesschau_results_df.columns
    ]
    tagesschau_inspection_table_df = tagesschau_results_df[inspection_cols].copy()
    tagesschau_inspection_table_df = tagesschau_inspection_table_df.sort_values(
        ["category", "row_id", "hit_text"],
        ascending=[True, True, True],
    ).reset_index(drop=True)

    display(tagesschau_label_summary_df)
    display(tagesschau_inspection_table_df)

    if not legacy_category_summary_df.empty:
        tagesschau_comparison_df = tagesschau_label_summary_df.copy()
        tagesschau_comparison_df.insert(0, "source", "Tagesschau")
        comparison_df = pd.concat([legacy_category_summary_df, tagesschau_comparison_df], ignore_index=True, sort=False)
        display(comparison_df)
else:
    print("No Tagesschau synchronous results file exists yet. Run the GPT cell first, then rerun this inspection cell.")

,category,coded_contexts,unique_articles,share_pct
0,NEUTRAL,66,49,98.51
1,POSITIONS-/PARTEILICHKEITS-BIAS,1,1,1.49


,row_id,source,Title,hit_text,category,evidence,context_window
0,14206,Tagesschau,Einigung im Streit um Kontrolle über Polizei,Deutschlandfunk,NEUTRAL,,"Laut Schwalb gibt das Gesetz aber nur her, das..."
1,14224,Tagesschau,"""Eine verfassungswidrige Invasion""",NDR Info,NEUTRAL,,"Und dann ergänzt er noch: ""Wir müssen unsere S..."
2,14251,Tagesschau,Die Fehler haben vor allem die Anderen gemacht,Deutschlandfunk,NEUTRAL,,"Das Buch ""107 Tage"" handelt von ungenutzten Mö..."
3,14251,Tagesschau,Die Fehler haben vor allem die Anderen gemacht,Politico,NEUTRAL,,"Buttigieg: ""War überrascht, als ich das las"" A..."
4,14336,Tagesschau,Trump will Antifa als Terrororganisation einst...,Deutschlandfunk,NEUTRAL,,Für Deutschland kommt das Bundesamt für Verfas...
...,...,...,...,...,...,...,...
62,20186,Tagesschau,"Fast 5,7 Millionen Erwachsene sind überschuldet",MDR,NEUTRAL,,"""Als Beispiel für eine Rückmeldung von Kliente..."
63,20277,Tagesschau,Deutsche Wirtschaft für besonnene Reaktion,NDR Info,NEUTRAL,,Einer Studie des Instituts der deutschen Wirts...
64,20353,Tagesschau,Dauerhaftes Single-Sein belastet,BR,NEUTRAL,,Eine Studie zeigt: Die Partnerlosigkeit hat Fo...
65,20421,Tagesschau,So erleben Wildtiere die Silvesternacht,SWR,NEUTRAL,,Gerade für Wildtiere ist die Böllerei besonder...


In [31]:
# show non-NEUTRAL Tagesschau context windows in full
non_neutral = tagesschau_inspection_table_df[tagesschau_inspection_table_df["category"] != "NEUTRAL"].copy()

if non_neutral.empty:
    print("No non-NEUTRAL Tagesschau contexts found.")
else:
    pd.set_option("display.max_colwidth", None)
    for _, row in non_neutral.iterrows():
        print("=" * 100)
        print(f"row_id: {row['row_id']}")
        print(f"Title: {row.get('Title', '')}")
        print(f"hit_text: {row.get('hit_text', '')}")
        print(f"category: {row['category']}")
        print(f"evidence:\n{row.get('evidence', '')}\n")
        print("context_window (full):")
        print(row.get("context_window", ""))
        print("=" * 100)

row_id: 17897
Title: Demonstranten fordern "Tod dem Diktator"
hit_text: Staatssender
category: POSITIONS-/PARTEILICHKEITS-BIAS
evidence:
Staatssender

context_window (full):
Immer stärker werden er und sein Regime von Iranerinnen und Iranern als Unterdrücker wahrgenommen. Landesweite Proteste, brennender Staatssender Unzählig viele Menschen gehen am Donnerstagabend im Iran auf die Straßen. Eine US-amerikanische Denkfabrik spricht von fast 160 Demonstrationen in 27 Provinzen des Landes.


## 6. Prepare the Full Batch Files Only After Inspection

This stays minimal on purpose. It writes the batch input and manifest for later, keeps `row_id` and the other matching metadata, and gives you a small cost table. It does not submit anything.

In [29]:
tagesschau_50_requests_df = build_batch_requests_df(
    tagesschau_50_context_df,
    prompt_template=CODEBOOK_PROMPT,
    model_name=MODEL_NAME,
    analysis_instructions=ANALYSIS_INSTRUCTIONS,
    frame_schema=FRAME_SCHEMA,
)
full_batch_requests_df = build_batch_requests_df(
    media_context_df,
    prompt_template=CODEBOOK_PROMPT,
    model_name=MODEL_NAME,
    analysis_instructions=ANALYSIS_INSTRUCTIONS,
    frame_schema=FRAME_SCHEMA,
)

for frame_name, frame in [("tagesschau_50", tagesschau_50_requests_df), ("full", full_batch_requests_df)]:
    validation_errors = validate_batch_requests(frame)
    if validation_errors:
        raise ValueError(f"{frame_name} batch validation failed:\n" + "\n".join(validation_errors[:20]))

write_batch_jsonl(tagesschau_50_requests_df, TAGESSCHAU_50_BATCH_JSONL_PATH)
write_manifest_csv(tagesschau_50_requests_df, TAGESSCHAU_50_MANIFEST_PATH)
write_batch_jsonl(full_batch_requests_df, FULL_BATCH_JSONL_PATH)
write_manifest_csv(full_batch_requests_df, FULL_MANIFEST_PATH)

BATCH_INPUT_PRICE_PER_MILLION = 0.25
BATCH_OUTPUT_PRICE_PER_MILLION = 2.00
OUTPUT_TOKEN_SCENARIOS = [40, 80]

cost_rows = []
for label, frame in [("tagesschau_50", tagesschau_50_requests_df), ("full", full_batch_requests_df)]:
    for output_tokens_per_request in OUTPUT_TOKEN_SCENARIOS:
        estimate = estimate_batch_cost(
            frame,
            output_tokens_per_request=output_tokens_per_request,
            batch_input_price_per_million=BATCH_INPUT_PRICE_PER_MILLION,
            batch_output_price_per_million=BATCH_OUTPUT_PRICE_PER_MILLION,
        )
        cost_rows.append(
            {
                "batch": label,
                "assumed_output_tokens_per_request": output_tokens_per_request,
                "requests": estimate.n_requests,
                "estimated_total_cost_usd": round(estimate.estimated_total_cost_usd, 4),
            }
        )

cost_estimate_df = pd.DataFrame(cost_rows)
print(f"Tagesschau 50 batch JSONL: {TAGESSCHAU_50_BATCH_JSONL_PATH}")
print(f"Tagesschau 50 manifest: {TAGESSCHAU_50_MANIFEST_PATH}")
print(f"Full batch JSONL: {FULL_BATCH_JSONL_PATH}")
print(f"Full manifest: {FULL_MANIFEST_PATH}")
display(cost_estimate_df)

Tagesschau 50 batch JSONL: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/media_framing_tagesschau_50_batch.jsonl
Tagesschau 50 manifest: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/media_framing_tagesschau_50_manifest.csv
Full batch JSONL: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/media_framing_full_batch.jsonl
Full manifest: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/media_framing_full_manifest.csv


,batch,assumed_output_tokens_per_request,requests,estimated_total_cost_usd
0,tagesschau_50,40,67,0.0330
1,tagesschau_50,80,67,0.0384
2,full,40,12009,6.0414
3,full,80,12009,7.0021
